# RNA Multiome: Subset Donors by Condition and Convert to AnnData

**Purpose:** Load Weston's full fNIH liver 10X multiome Seurat object,
split it by `condition`, and convert each condition's RNA counts to an
`.h5ad` (AnnData) file for use as a Tangram reference.

**Performance notes:** loading the full `.rds` object took about an hour
with 200 GB of memory in one run, and about 8 minutes with 700 GB in
another -- worth keeping in mind when picking node resources for this
notebook.

**Cleanup notes (this pass):** removed a duplicate `library(Seurat)` call,
a stale commented-out `SplitObject(sobj_ds, ...)` line referencing an object
that doesn't exist in this notebook (`sobj_ds`), a duplicate re-run of the
`SplitObject(...)` call (with an in-between cell that only printed a
timestamp), and a stray leading space causing inconsistent indentation.
`dir.create()` now passes `showWarnings = FALSE` so re-running the notebook
doesn't warn/error when the output directory already exists.

In [10]:
suppressMessages(library(Seurat))
suppressMessages(library(SeuratDisk))
suppressMessages(library(reticulate))  # used implicitly by SaveH5Seurat/Convert below
suppressMessages(library(Signac))      # needed for the multiome object's ChromatinAssay methods,
                                        # even though no Signac:: functions are called directly


## Load the full multiome object

In [1]:
print(Sys.time())
sobj <- readRDS('/tscc/lustre/ddn/scratch/welison/Liver_Share/result.depot/Final_fNIH_Liver_10X_Multiome_object_v3_Gaulton_Peaks_TSCC_Fragments.rds')
print(Sys.time())


[1] "2025-01-31 09:27:11 PST"
[1] "2025-01-31 09:36:38 PST"


In [2]:
sobj

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.3; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: Seurat

Loading required package: Signac



An object of class Seurat 
1160060 features across 408178 samples within 5 assays 
Active assay: RNA (36601 features, 0 variable features)
 2 layers present: counts, data
 4 other assays present: SCT, signac.peaks, Peaks.1, Peaks.2
 5 dimensional reductions calculated: pca, harmony.rna, umap.rna, umap.atac, umap.wnn

In [4]:
colnames(sobj@meta.data)

[1] "orig.ident"                        "nCount_RNA"                       
 [3] "nFeature_RNA"                      "percent.mt"                       
 [5] "nCount_RNA_raw"                    "nFeature_RNA_raw"                 
 [7] "donor_demux"                       "nCount_SCT"                       
 [9] "nFeature_SCT"                      "SCT.weight"                       
[11] "seurat_clusters"                   "log_nCount_SCT"                   
[13] "log_nFeature_SCT"                  "lane"                             
[15] "batch"                             "gex_raw_reads"                    
[17] "gex_mapped_reads"                  "gex_conf_intergenic_reads"        
[19] "gex_conf_exonic_reads"             "gex_conf_intronic_reads"          
[21] "gex_conf_exonic_unique_reads"      "gex_conf_exonic_antisense_reads"  
[23] "gex_conf_exonic_dup_reads"         "gex_exonic_umis"                  
[25] "gex_conf_intronic_unique_reads"    "gex_conf_intronic_antisense_reads"
[27] "gex_conf_intronic_dup_reads"       "gex_intronic_umis"                
[29] "gex_conf_txomic_unique_reads"      "gex_umis_count"                   
[31] "gex_genes_count"                   "atac_raw_reads"                   
[33] "atac_unmapped_reads"               "atac_lowmapq"                     
[35] "atac_dup_reads"                    "atac_chimeric_reads"              
[37] "atac_mitochondrial_reads"          "atac_fragments"                   
[39] "atac_TSS_fragments"                "atac_peak_region_fragments"       
[41] "atac_peak_region_cutsites"         "TSS.enrichment"                   
[43] "TSS.percentile"                    "condition"                        
[45] "disease_status"                    "library"                          
[47] "amulet"                            "nCount_Peaks"                     
[49] "nFeature_Peaks"                    "Description"                      
[51] "Age"                               "BMI"                              
[53] "Gender"                            "Alcoholic..2..Drinks.Day."        
[55] "Steatosis.grade"                   "Steatosis.."                      
[57] "Fat.distribution"                  "Lobular.inflammation"             
[59] "Ballooning"                        "Portal.inflammation"              
[61] "Hepatocyte.necrosis"               "MASH.CRN.score..X.8"              
[63] "Diagnosis"                         "Peaks.weight"                     
[65] "value"                             "barcode"                          
[67] "cellsubtype"                       "celltype"                         
[69] "nCount_Peaks_Full"                 "nFeature_Peaks_Full"              
[71] "Fibrosis.stage"                    "nCount_Peaks.1"                   
[73] "nFeature_Peaks.1"                  "nCount_Peaks.2"                   
[75] "nFeature_Peaks.2"

In [5]:
table(sobj$condition)


Control    MASH    MASL  MetALD 
 131993   88203  131832   56150 

## Single-condition export example (MetALD)

Saving just one condition (MetALD) took under a minute; kept here as a quick example/sanity check before the full per-condition loop below.

In [6]:
print(Sys.time())
seurat_obj <- subset(sobj, subset = condition == 'MetALD')
print(Sys.time())

[1] "2025-01-31 10:09:02 PST"
[1] "2025-01-31 10:09:39 PST"


In [7]:
odir_spl <- '/tscc/projects/ps-epigen/users/cmiciano/Liver/NASH_NAFLD_pooling/outputs/sandbox/rna_multiome_cond/'

In [8]:
dir.create(odir_spl, showWarnings = FALSE)


In [11]:
print(Sys.time())

# Convert all metadata columns to character (avoids type-mismatch issues on export)
seurat_obj@meta.data[] <- lapply(seurat_obj@meta.data, as.character)

# Set the default assay to RNA
DefaultAssay(seurat_obj) <- "RNA"

sample_name <- 'MetALD'

h5seurat_filename <- paste0(odir_spl, sample_name, ".h5Seurat")
print(h5seurat_filename)

h5ad_filename <- paste0(odir_spl, sample_name, ".h5ad")
print(h5ad_filename)

# Save the Seurat object as an H5Seurat file, then convert to .h5ad (AnnData)
SaveH5Seurat(seurat_obj, filename = h5seurat_filename, overwrite = TRUE)
Convert(h5seurat_filename, dest = "h5ad", assay = "RNA", slot = "counts", overwrite = TRUE)

print(Sys.time())


[1] "2025-01-31 10:53:16 PST"
[1] "/tscc/projects/ps-epigen/users/cmiciano/Liver/NASH_NAFLD_pooling/outputs/sandbox/rna_multiome_cond/MetALD.h5Seurat"
[1] "/tscc/projects/ps-epigen/users/cmiciano/Liver/NASH_NAFLD_pooling/outputs/sandbox/rna_multiome_cond/MetALD.h5ad"


Creating h5Seurat file for version 3.1.5.9900

Adding counts for RNA

Adding data for RNA

No variable features found for RNA

No feature-level metadata found for RNA

Adding counts for SCT

Adding data for SCT

Adding scale.data for SCT

Adding variable features for SCT

No feature-level metadata found for SCT

Writing out SCTModel.list for SCT

Adding counts for signac.peaks

Adding data for signac.peaks

No variable features found for signac.peaks

Adding feature-level metadata for signac.peaks

Writing out ranges for signac.peaks

Writing out motifs for signac.peaks

Writing out fragments for signac.peaks

Writing out seqinfo for signac.peaks

Writing out annotation for signac.peaks

Writing out bias for signac.peaks

Writing out positionEnrichment for signac.peaks

Writing out links for signac.peaks

Adding counts for Peaks.1

Adding data for Peaks.1

No variable features found for Peaks.1

Adding feature-level metadata for Peaks.1

Writing out ranges for Peaks.1

Writing out moti

[1] "2025-01-31 10:59:30 PST"


## Split by condition and export every condition

In [ ]:
print(Sys.time())
split_objects <- SplitObject(sobj, split.by = "condition")
print(Sys.time())


[1] "2025-01-31 12:16:36 PST"


In [13]:
# NOTE: displaying split_objects errored previously (likely just from the
# size/complexity of the repr for a list of large Seurat objects, not a
# processing error) -- kept as a sanity check, but don't be alarmed if it
# errors or takes a while.
split_objects


ERROR: Error in eval(expr, envir, enclos): object 'split_objects' not found


In [ ]:
# Loop through each split object (one per condition), process it, and save it.
for (sample_name in names(split_objects)) {
  print(Sys.time())
  seurat_obj <- split_objects[[sample_name]]

  # Convert all metadata columns to character (avoids type-mismatch issues on export)
  seurat_obj@meta.data[] <- lapply(seurat_obj@meta.data, as.character)

  # Set the default assay to RNA
  DefaultAssay(seurat_obj) <- "RNA"

  sample_name <- gsub("[[:punct:][:space:]]", "_", sample_name)

  h5seurat_filename <- paste0(odir_spl, sample_name, ".h5Seurat")
  print(h5seurat_filename)

  h5ad_filename <- paste0(odir_spl, sample_name, ".h5ad")
  print(h5ad_filename)

  # Save the Seurat object as an H5Seurat file, then convert to .h5ad (AnnData)
  SaveH5Seurat(seurat_obj, filename = h5seurat_filename, overwrite = TRUE)
  Convert(h5seurat_filename, dest = "h5ad", assay = "RNA", slot = "counts", overwrite = TRUE)

  print(Sys.time())
}
